# Voice-worker в Google Colab

**Как открыть:** загрузите этот файл `colab.ipynb` в [Google Colab](https://colab.research.google.com/) через **Файл → Загрузить блокнот**, либо положите репозиторий на Drive и откройте ноутбук оттуда.

Подробности см. **`COLAB.md`** в том же репозитории, что и этот ноутбук.

На Colab нужен **только** каталог **voice-worker** (Python). Go-gateway и веб-интерфейс запускаются локально на ПК.

**Код и база хранятся на Google Drive** — ничего не теряется между сессиями.

Структура на Drive (пример):
```
MyDrive/
└── mafia-voice/
    ├── voice-worker/          ← скопируйте сюда только services/voice-server/voice-worker
    └── voice_registry.sqlite  ← опционально (или задаётся VOICE_SERVER_DB)
```

> Скопируйте папку **voice-worker** из репозитория на Drive **один раз**.
> При каждом запуске — ячейки ниже по порядку.


In [ ]:
# @title Шаг 1: Монтирование Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

VOICE_WORKER_ROOT = '/content/drive/MyDrive/mafia-voice/voice-worker'

if os.path.isdir(VOICE_WORKER_ROOT):
    os.chdir(VOICE_WORKER_ROOT)
    print(f"✅ Рабочая директория: {os.getcwd()}")
else:
    print(f"❌ ОШИБКА: Путь {VOICE_WORKER_ROOT} не найден!")

In [ ]:
# @title Шаг 2: Умная установка зависимостей (CPU / GPU)
import subprocess
import shutil
import sys

def prepare_environment():
    print("🔍 Анализ окружения...")

    # 1. Удаляем конфликтные пакеты
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)

    # 2. Проверяем наличие GPU через системную утилиту
    has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
        ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode == 0

    if has_gpu:
        print("🚀 GPU обнаружен. Устанавливаем стандартные зависимости...")
        # Ставим всё как есть, pip подтянет CUDA-версии
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    else:
        print("🐌 GPU не найден. Оптимизация под CPU...")
        # Сначала сносим тяжелый предустановленный torch
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchaudio"], check=False)

        # Ставим легковесный CPU-torch
        print("📥 Установка torch+cpu (это сэкономит ~3ГБ места)...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cpu"
        ])

        # Доставляем остальные зависимости из файла
        print("📥 Установка остальных пакетов...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

    print("\n✅ Готово! ⚠ ОБЯЗАТЕЛЬНО: Выполните 'Runtime' -> 'Restart session' (Среда выполнения -> Перезапустить сессию)")

prepare_environment()

In [ ]:
# @title Шаг 3: Конфиг на Google Drive (YAML) + секреты
import os

import yaml

# Снова переходим в папку (на случай если рестарт сбросил путь)
VOICE_WORKER_ROOT = '/content/drive/MyDrive/mafia-voice/voice-worker'
os.chdir(VOICE_WORKER_ROOT)

DB_DIR = '/content/drive/MyDrive/mafia-voice'
os.makedirs(DB_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(DB_DIR, 'voice-server-config.yaml')

if not os.path.isfile(CONFIG_PATH):
    base_cfg = {
        'api_key': 'barchik',
        'max_file_size_mb': 500,
        'db_path': os.path.join(DB_DIR, 'voice_registry.sqlite'),
        'use_google_drive': True,
        'whisper_model': 'large-v2',
        'voice_threshold_preset': 'balanced',
        'enable_voice_learning': True,
        'embedding_backend': 'wespeaker',
    }
    with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
        yaml.safe_dump(base_cfg, f, allow_unicode=True, sort_keys=False)
    print(f'📝 Создан шаблон конфига: {CONFIG_PATH}')
else:
    print(f'📂 Используется конфиг: {CONFIG_PATH}')

os.environ['VOICE_SERVER_CONFIG_PATH'] = CONFIG_PATH

# Каждую сессию Colab: актуальное устройство (перекрывает поле device в YAML при необходимости)
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['VOICE_SERVER_DEVICE'] = device

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

assert os.environ.get('HF_TOKEN'), 'Задайте HF_TOKEN в Colab Secrets (иконка ключа)!'

print(f'🎯 Устройство (env): {device.upper()}')
print(f'⚙️ VOICE_SERVER_CONFIG_PATH={CONFIG_PATH}')

In [ ]:
# @title Шаг 3.5: Установка ngrok через apt
import subprocess

install_script = """
curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | tee /etc/apt/trusted.gpg.d/ngrok.asc > /dev/null \
  && echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' \
  | tee /etc/apt/sources.list.d/ngrok.list \
  && apt-get update -q \
  && apt-get install -yq ngrok
"""

r = subprocess.run(["bash", "-c", install_script], capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else "")
print(r.stderr[-200:] if r.stderr else "")

# Проверяем
r2 = subprocess.run(["ngrok", "version"], capture_output=True, text=True)
print(f"\n{'✅' if r2.returncode == 0 else '❌'} {r2.stdout.strip() or r2.stderr.strip()}")

### Шаг 4: где именно запускается Whisper / voice-worker

**Сервис поднимается в следующей ячейке с кодом** (не в отдельном терминале).

Цепочка такая:

1. **`from app.main import app`** — загружается приложение FastAPI из файла **`voice-worker/app/main.py`** (пакет `app` на диске в `VOICE_WORKER_ROOT`).
2. **`uvicorn.Server` + `await server.serve()`** — запускается **HTTP-сервер на порту 8000**. Это и есть «виспер-сервис»: он принимает запросы (`/process_chunk`, `/reset`, …).
3. **Whisper / pyannote** вызываются **внутри обработчиков** при обращении к API (модели часто подгружаются при **первом** запросе с аудио, см. `app/pipeline.py`), а не в момент старта uvicorn.
4. **ngrok** пробрасывает порт 8000 в интернет — по напечатанному `https://…` вы потом указываете **`-voice-url`** в Go на своём ПК.

Ячейка будет выполняться долго («висеть») — пока работает сервер. Остановка: **Stop** или Runtime → Interrupt.


In [ ]:
# @title Шаг 4: Запуск сервера (ngrok + uvicorn)
import sys
import logging

# Сбрасываем кэш модулей app — подхватит любые изменения в папке app/
for key in list(sys.modules.keys()):
    if key == "app" or key.startswith("app."):
        del sys.modules[key]

# Настраиваем логирование ДО импорта app
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

import nest_asyncio
nest_asyncio.apply()

sys.path.insert(0, VOICE_WORKER_ROOT)

import subprocess
from google.colab import userdata
import uvicorn
from app.main import app  # импорт после сброса кэша и настройки логов

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
if not NGROK_TOKEN:
    print("❌ ОШИБКА: Задайте NGROK_TOKEN в Secrets!")
else:
    r = subprocess.run(["ngrok", "config", "add-authtoken", NGROK_TOKEN],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"❌ Ошибка authtoken: {r.stderr.strip()}")
    else:
        tunnel_proc = subprocess.Popen(
            ["ngrok", "http", "8000", "--log", "stdout", "--log-format", "json"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )

        import json, time
        public_url = None
        for _ in range(30):
            line = tunnel_proc.stdout.readline()
            try:
                entry = json.loads(line)
                if entry.get("msg") == "started tunnel":
                    public_url = entry["url"]
                    break
            except:
                pass
            time.sleep(0.3)

        if public_url:
            print(f"\n🔗 Публичный URL для Gateway: {public_url}")
            print("🚀 Сервер запускается...")
        else:
            print("⚠️ Не удалось получить URL, но сервер запускается...")

        config = uvicorn.Config(
            app,
            host="0.0.0.0",
            port=8000,
            log_level="info",
            log_config=None,  # не даём uvicorn перезаписывать наш logging
        )
        server = uvicorn.Server(config)
        await server.serve()

Чтобы остановить сервер: **прервите выполнение** ячейки выше (кнопка Stop) или Runtime → Interrupt.